In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df_client = pd.read_csv("data/customers_clean.csv")
df_transaction = pd.read_csv("data/transaction_clean.csv")
df_transaction = df_transaction[df_transaction["from_store"]== "yep"]

# Répartition des clients par pays
country_dist = df_client['country'].value_counts()
country_pct = (country_dist / len(df_client) * 100).round(2)

print("=== RÉPARTITION DES CLIENTS PAR PAYS ===")
print(pd.DataFrame({'Nombre': country_dist, 'Pourcentage': country_pct}))

# Comportements d'achat par pays
behavior_by_country = df_client.groupby('country').agg({
    'avg_basket': 'mean',
    'n_orders': 'mean',
    'total_spent': 'mean'
}).round(2)

print("\n=== COMPORTEMENTS D'ACHAT PAR PAYS ===")
print(behavior_by_country)

# PCA
print("\n=== ANALYSE EN COMPOSANTES PRINCIPALES ===")
numeric_cols = ['n_orders', 'total_spent', 'avg_basket', 'recency_days', 'tenure_days']
X = df_client[numeric_cols].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print("\nVariance expliquée par composante:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {var:.2%}")
print(f"\nVariance cumulée (2 premières): {pca.explained_variance_ratio_[:2].sum():.2%}")

# Contributions des variables
print("\n=== COMPOSANTES PRINCIPALES (LOADINGS) ===")
components_df = pd.DataFrame(
    pca.components_[:2].T,
    columns=['PC1', 'PC2'],
    index=numeric_cols
).round(3)
print(components_df)

print("\n=== INTERPRÉTATION ===")
print("PC1 - Variables dominantes:")
pc1_abs = components_df['PC1'].abs().sort_values(ascending=False)
for var in pc1_abs.head(3).index:
    print(f"  {var}: {components_df.loc[var, 'PC1']:.3f}")

print("\nPC2 - Variables dominantes:")
pc2_abs = components_df['PC2'].abs().sort_values(ascending=False)
for var in pc2_abs.head(3).index:
    print(f"  {var}: {components_df.loc[var, 'PC2']:.3f}")

# Visualisations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Top 10 pays par nombre de clients
top_countries = country_dist.head(10)
axes[0, 0].bar(range(len(top_countries)), top_countries.values)
axes[0, 0].set_xticks(range(len(top_countries)))
axes[0, 0].set_xticklabels(top_countries.index, rotation=45, ha='right')
axes[0, 0].set_title('Top 10 pays par nombre de clients')
axes[0, 0].set_ylabel('Nombre de clients')

# Panier moyen par pays (top 10)
top_basket = behavior_by_country.nlargest(10, 'avg_basket')
axes[0, 1].barh(range(len(top_basket)), top_basket['avg_basket'])
axes[0, 1].set_yticks(range(len(top_basket)))
axes[0, 1].set_yticklabels(top_basket.index)
axes[0, 1].set_title('Top 10 pays par panier moyen')
axes[0, 1].set_xlabel('Panier moyen')

# Fréquence d'achat par pays (top 10)
top_freq = behavior_by_country.nlargest(10, 'n_orders')
axes[1, 0].barh(range(len(top_freq)), top_freq['n_orders'])
axes[1, 0].set_yticks(range(len(top_freq)))
axes[1, 0].set_yticklabels(top_freq.index)
axes[1, 0].set_title('Top 10 pays par fréquence d\'achat')
axes[1, 0].set_xlabel('Nombre moyen de commandes')

# Scree plot
axes[1, 1].bar(range(1, len(pca.explained_variance_ratio_) + 1), 
               pca.explained_variance_ratio_, alpha=0.6, label='Individuelle')
axes[1, 1].plot(range(1, len(pca.explained_variance_ratio_) + 1), 
                pca.explained_variance_ratio_.cumsum(), 'r-o', label='Cumulée')
axes[1, 1].axhline(y=0.8, color='g', linestyle='--', alpha=0.5, label='80%')
axes[1, 1].set_xlabel('Composante')
axes[1, 1].set_ylabel('Variance expliquée')
axes[1, 1].set_title('Scree Plot - Variance expliquée par composante')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_xticks(range(1, len(pca.explained_variance_ratio_) + 1))

plt.tight_layout()
plt.show()

# Biais géographique
uk_pct = country_pct.get('United Kingdom', 0)
print(f"\n=== BIAIS GÉOGRAPHIQUE ===")
print(f"Pays dominant: {country_dist.index[0]} ({country_pct.iloc[0]}%)")
print(f"Top 3 pays représentent: {country_pct.head(3).sum()}% des clients")
print(f"Nombre total de pays: {len(country_dist)}")